In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightkurve as lk

import glob 
from tqdm import tqdm
from copy import deepcopy

from astropy.stats import sigma_clipped_stats

from sklearn.cluster import DBSCAN

from Kakapo.photometry import forced_photometry
# from Kakapo.difference_image import create_diff_image
from Kakapo.difference_image import Difference_Imaging
from Kakapo.cleaning_curve import wavelet_denoise, correction_smoothing_lightcurve, flatten_and_clip_outliers

%matplotlib widget

In [ ]:
def get_valid_baseline_indices(lc, frame_start, frame_end, base_range, min_points=55):
    """
    Expands the baseline region until at least `min_points` non-NaN values are found.
    
    Parameters:
        lc : array-like
            The lightcurve.
        frame_start : int
            Start of the event window.
        frame_end : int
            End of the event window.
        base_range : int
            Initial extension from event to define baseline.
        min_points : int
            Minimum number of non-NaN values required in the baseline.
    
    Returns:
        baseline_start : int
            Final start index of baseline region.
        baseline_end : int
            Final end index of baseline region.
        valid_inds : np.ndarray (bool)
            Boolean array marking valid baseline indices (non-NaN).
    """
    baseline_start = frame_start - base_range
    baseline_end = frame_end + base_range

    max_len = len(lc)
    frames = np.arange(max_len)

    while True:
        baseline_start = max(0, baseline_start)
        baseline_end = min(max_len, baseline_end)

        baseline_mask = ((frames > baseline_start) & (frames < frame_start)) | \
                        ((frames < baseline_end) & (frames > frame_end))
        valid_inds = baseline_mask & ~np.isnan(lc)

        if np.sum(valid_inds) >= min_points or (baseline_start == 0 and baseline_end == max_len):
            break

        baseline_start -= 1
        baseline_end += 1

    return baseline_start, baseline_end, baseline_mask

def _check_lc_significance(diff, distance, start, end, x, y, flux_sign, buffer = 1.1, base_range=2.85, grad_val = -40):
    cadence = 0.5/24
    
    buffer = int(buffer/cadence)
    base_range = int(base_range/cadence)
    
    lc = forced_photometry(diff, x, y, None)
    lc = correction_smoothing_lightcurve(lc, distance < 0.25, window=35, sigma=3)
    lc = wavelet_denoise(lc, wavelet='coif5', level=3, keep='low', mode = 'smooth')
    
    # if len(self.thrusters) > 10:
    #     lc = correct_motion_lightcurve(lc, self.distance, self.thrusters)
    
    gradients = np.gradient(lc)
    
    # Setting up the light curve
    frame_start = start - buffer
    frame_end = end + buffer
    if frame_start < 0:
        frame_start = 0
        frame_end += buffer
    if frame_end > len(lc):
        frame_end = len(lc) - 1 
        frame_start -= buffer
    
    if (frame_start < 0):
        frame_start = 0
    if (frame_end > len(lc)):
        frame_end = len(lc) - 1 
    
    baseline_start = frame_start - base_range
    baseline_end = frame_end + base_range
    if baseline_start < 0:
        baseline_start = 0
    if baseline_end > len(lc):
        baseline_end = len(lc) - 1
    
    frames = np.arange(len(lc))
    
    baseline_start, baseline_end, ind = get_valid_baseline_indices(lc, frame_start, frame_end, base_range, min_points=48)
    med = np.nanmedian(lc[ind])
    gradmed = np.nanmedian(gradients[ind])
    std = np.nanstd(lc[ind], ddof = 1)
    gradstd = np.nanstd(gradients[ind], ddof = 1)
    lcevent = lc[int(start):int(end)]
    gradevent = gradients[int(start):int(end)]
    
    lc_sig = (lcevent - med) / std
    grad_sig = (gradients - gradmed) / gradstd
    
    indices = np.where((np.abs(grad_sig) < 8) & (gradients > grad_val))[0]
    
    try:
        sig_max = abs(np.nanmax(lc_sig))
        sig_med = abs(np.nanpercentile(lc_sig, 84))
    except:
        sig_med = -1
        sig_max = -1
    
    # print('Frame Start:', frame_start, frame_end, type(frame_start), type(frame_end))
    
    if np.nansum(np.isfinite(lc[int(frame_start):int(frame_end)])) < 5:
        sig_med = -1
    
    lc_sig = (lc - med) / std
    return sig_max, sig_med, lc_sig * flux_sign, indices

In [ ]:
def detected_events(diff, distance, events, siglim = 2):
    
    cluster_ids = np.unique(events['cluster'])
    
    full_events = pd.DataFrame(columns=['cluster', 'sig_max', 'sig_84', 'frame_min', 'frame_max'])
    
    new_stars = pd.DataFrame(columns=events.columns)
    
    events['lc_sig'] = -1*np.ones(len(events))
    
    for cluster_id in tqdm(cluster_ids, desc= 'Clusters'):
        cluster = events[events['cluster'] == cluster_id]
        
        if len(cluster) < 5:
            continue
        
        frame_min = cluster['frame'].min()
        frame_max = cluster['frame'].max()
        
        x, _, xstd = sigma_clipped_stats(cluster['xcentroid'].values, sigma = 3)
        y, _, ystd = sigma_clipped_stats(cluster['ycentroid'].values, sigma = 3)
        
        if (xstd >= 0.8) | (ystd >= 0.8):
            continue
        
        sig_max, sig_med, lc_sig, indices = _check_lc_significance(diff, distance, frame_min, frame_max, x, y, 1, 
                                                                        buffer = 1.2, base_range=2.6, grad_val = -60)
        if sig_med < siglim:
            # print('ZZZ Sig. Med', sig_med)
            continue
        
        filtered_cluster = cluster[cluster['frame'].isin(indices)]
        filtered_cluster = filtered_cluster.reset_index(drop=True)
        frame_inds = filtered_cluster['frame'].values.astype(int)
        
        filtered_lc_sig = lc_sig[frame_inds]
        filtered_lc_sig_indices = filtered_lc_sig > siglim
        
        filtered_frame_values = filtered_cluster['frame'].values[filtered_lc_sig_indices]
        filtered_frame_values = np.sort(filtered_frame_values, axis=None) 
        
        filtered_lc_sig = filtered_lc_sig[filtered_lc_sig_indices]
        filtered_final_cluster = filtered_cluster[filtered_cluster['frame'].isin(filtered_frame_values)]
        filtered_final_cluster['lc_sig'] = filtered_lc_sig
        
        if len(filtered_lc_sig) < 5:
            continue
        else:
            # print(len(filtered_lc_sig))
            filtered_cluster['cluster'] = len(full_events)
            new_stars = pd.concat([new_stars, filtered_cluster])
            full_events.loc[len(full_events)] = [len(full_events), sig_max, sig_med, 
                                                    filtered_final_cluster['frame'].min(), 
                                                    filtered_final_cluster['frame'].max()]
    
    new_stars = new_stars.reset_index(drop=True)
    
    # if len(full_events) == 0:
    #     # print('ZZZ We have failure!')
    #     return None, None
    # else:
    #     full_events, events_filtered = _lightcurve_event_checker(new_stars, full_events)

    return full_events 

In [ ]:
def _tpf_addition(tpf_info, tpf_input):
    
    if tpf_input.campaign is None:
        campaign = tpf_input.quarter
        mission = 'Kepler'
        print(f"Adding TPF {tpf_input.targetid} from {mission} quarter {campaign}")
    else:
        campaign = tpf_input.campaign
        mission = 'K2'
    
    tpf_info.loc[len(tpf_info)] = [mission, campaign, tpf_input.targetid, tpf_input.ra, tpf_input.dec, 
                                   tpf_input.flux.value, tpf_input.flux_err.value, tpf_input.quality, 
                                   tpf_input.pos_corr1, tpf_input.pos_corr2, tpf_input.time]
    
    return tpf_info

def _check_tpf_type(tpf_input):
    
    tpf_info = pd.DataFrame(columns=['mission', 'campaign', 'targetid', 'ra', 'dec', 'flux', 'flux_err', 
                                     'quality', 'pos_corr1', 'pos_corr2', 'time'])
        
    for i in range(len(tpf_input)):
        if isinstance(tpf_input[i], lk.targetpixelfile.KeplerTargetPixelFile):
            tpf_info = _tpf_addition(tpf_info, tpf_input[i])
        
    return tpf_info

In [ ]:
def _grouping(corr, f_dist = 12):
    
    def custom_distance(p1, p2):
        xy_dist = np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2) # Euclidean distance for xcentroid and ycentroid
        frame_dist = np.abs(p1[2] - p2[2]) # Absolute difference for frame
        
        if xy_dist <= 1 and frame_dist <= f_dist: # Combine both distances with their respective thresholds
            return 0  # In the same cluster (distance 0 means they are close enough)
        else:
            return 5  # Distance larger than threshold, separate clusters

    try:
        data = corr[['xcentroid', 'ycentroid', 'frame']].values

        db = DBSCAN(eps=1, min_samples=5, metric=custom_distance)  # 'metric' could be adjusted for your use case
        corr['cluster'] = db.fit_predict(data)
        
        corr = corr[corr['cluster'] != -1]
        
    except:
        return None
    
    return corr

def initial_filter(df):
    df = df[(df.fwhm > 0.8) & (df.snr >= 4) & 
            (df.psfdiff <= 2) & (df.poisson_thresh >= 1) & 
            (abs(df.correlation) >= 0.05)]
    
    return df

def mask_detections(correlation, psfdiff, fwhm, snr, 
                    roundness, poisson_thresh, xstd, ystd):
    
    mask =  (correlation >= 0.05) & (psfdiff <= 1.5) & \
            (fwhm <= 5) & (fwhm >= 0.95) & \
            (snr >= 4) & (snr < 10000) & (abs(roundness) <= 0.8) & \
            (poisson_thresh >= 1) & \
            (xstd <= 0.5) & (ystd <= 0.5)
    
    print(f'mask: {mask}')
    
    return ~mask

In [ ]:
def access_tpfs():
    """
    """

    test_case = []

    lightkurve_file_folder = '/Users/zgl12/.lightkurve/cache/mastDownload/K2/'

    files = sorted(glob.glob(lightkurve_file_folder + '*205922648*/*.fits.gz'))

    for file in tqdm(files, desc='Reading TPFs'):
        tpf = lk.read(file, quality_bitmask = 'none')
        test_case.append(tpf)
        
    return test_case

In [ ]:
tpf = access_tpfs()
tpf_info = _check_tpf_type(tpf)
tpf_info

In [ ]:
csv_file_1 = '/Users/zgl12/Modules/Kakapo/Data/csv_files/c3/c3_t205922648.csv'
# csv_file_1 = '/Users/zgl12/Modules/Kakapo/Data/csv_files/c16/c16_t211553428.csv'
# csv_file_1 = '/Users/zgl12/Modules/Kakapo/temp_mem/c16_t211553428.csv'
# csv_file_2 = '/Users/zgl12/Modules/Kakapo/temp_mem/c3_t206010203.csv'

# /Users/zgl12/Modules/Kakapo/Data/figures/c16/figures_c16_t211553428_e1.png

In [ ]:
df_1 = pd.read_csv(csv_file_1)
# df_2 = pd.read_csv(csv_file_2)

df_1 = initial_filter(df_1)
# df_2 = initial_filter(df_2)

df_1 = _grouping(df_1, f_dist = 48)
# df_2 = _grouping(df_2, f_dist = 18)

In [ ]:
plt.figure()
plt.scatter(df_1.xcentroid, df_1.ycentroid, c = df_1.frame)
plt.title(r'$x$ and $y$')
plt.xlabel(r'$x$')
plt.ylabel(r'$y$')
plt.show()

plt.figure()
plt.scatter(df_1.fwhm, df_1.correlation, c = df_1.frame)
plt.title(r'fwhm and correlation')
plt.xlabel(r'fwhm')
plt.ylabel(r'correlation')
plt.show()

plt.figure()
plt.scatter(df_1.frame, df_1.snr, c = df_1.frame)
plt.title(r'frame and snr')
plt.xlabel(r'frame')
plt.ylabel(r'snr')
plt.show()

In [ ]:
for cluster in np.unique(df_1.cluster.values):
    
    temp_df = df_1[df_1.cluster == cluster]
    print(cluster)
    print(len(temp_df), temp_df.frame.min(), temp_df.frame.max())
    x, _, xstd = sigma_clipped_stats(temp_df.xcentroid.values, sigma = 3)
    y, _, ystd = sigma_clipped_stats(temp_df.ycentroid.values, sigma = 3)
    
    print(f"{x:.2f} +/- {xstd:.2f}")
    print(f"{y:.2f} +/- {ystd:.2f}")
    print()

In [ ]:
temp_df_1 = df_1[df_1.cluster == 8]
# temp_df_2 = df_2[df_2.cluster == 1]

In [ ]:
x, _, xstd = sigma_clipped_stats(temp_df_1.xcentroid.values, sigma = 3)
y, _, ystd = sigma_clipped_stats(temp_df_1.ycentroid.values, sigma = 3)
correlation, _, _ = sigma_clipped_stats(temp_df_1.correlation.values, sigma = 3)
psfdiff, _, _  = sigma_clipped_stats(temp_df_1.psfdiff.values, sigma = 3)
snr, _, _  = sigma_clipped_stats(temp_df_1.snr.values, sigma = 3)
fwhm, _, _  = sigma_clipped_stats(temp_df_1.fwhm.values, sigma = 3)
roundness, _, _  = sigma_clipped_stats(temp_df_1.roundness.values, sigma = 3)
poisson_thresh, _, _  = sigma_clipped_stats(temp_df_1.poisson_thresh.values, sigma = 3)

mask_detections(correlation, psfdiff, fwhm, snr, 
                    roundness, poisson_thresh, xstd, ystd)

# x_2, _, xstd = sigma_clipped_stats(temp_df_2.xcentroid.values, sigma = 3)
# y_2, _, ystd = sigma_clipped_stats(temp_df_2.ycentroid.values, sigma = 3)

In [ ]:
print(correlation)
print(psfdiff)
print(snr)
print(fwhm)
print(roundness)
print(poisson_thresh)

In [ ]:
# # diff = np.load('/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t205922648.npy')
# # diff = np.load('/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t206010203.npy')
# # diff = np.load('/Users/zgl12/Modules/Kakapo/temp_mem/diff_c16_t211553428.npy')

epsf = np.genfromtxt('/Users/zgl12/Modules/Kakapo/epsf_data.txt')
epsf = epsf[2:-2,2:-2]

# # ref, diffs, poisson_noise, thrusters, distance = create_diff_image(tpf_info.iloc[0], epsf, tol=0.2)

chunky_bird = Difference_Imaging(tpf_info.iloc[0], epsf, tol=0.2)

ref = chunky_bird.ref
noise = chunky_bird.noise
diffs = chunky_bird.diffs
thrusters = chunky_bird.thrusters
distance = chunky_bird.distance

# # print(ref)


# # New background and noise check
#     # Gaussian Filter
#     # Polynomial background
#     # Before minimize
#     # After minimize
# # Combined weighting of both ind and mass
#     # Weight ind
#     # Weight mass
#     # Weight both
# # Flatten and check multiple massive peaks
#     # 
# # Flatten with a Gaussian process and then clip outliers
#     # Currently in progress

# # Gaussian, After, Weight ind
# # Gaussian, After, Weight mass
# # Gaussian, After, Weight both
# # Polynomial, After, Weight both

# # diff2 = np.load('/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t206010203.npy')

In [ ]:
events = detected_events(diffs, distance, df_1, siglim = 0.1)

In [ ]:
events

In [ ]:
# diffs = np.load('/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t205922648.npy')
# diffs = np.load('/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c16/diff_c16_t211553428.npy')
# diffs = np.load('')
# distance = np.genfromtxt('/Users/zgl12/Modules/Kakapo/temp_mem/temp_distance.txt')

# np.save('/Users/zgl12/Modules/Kakapo/temp_mem/temp_diffIm.npy', diffs)
# np.savetxt('/Users/zgl12/Modules/Kakapo/temp_mem/temp_distance.txt', distance)

In [ ]:

# flux_psf = forced_photometry(diff_shifted, x, y, epsf, bkg = True, method = 'psf')
flux_ape = forced_photometry(diffs, x, y, None, bkg = True, method = 'aperture')

wavelet = 'coif5'
mode = 'smooth'
levels = [2, 3]

# flux_ape, flattened, mask = flatten_and_mask_outliers(flux_ape, gp_scale=40000, outlier_sigma=5.0)
# flux_ape[mask] = np.nan

# flat_corr_flux = flux_ape - flattened + flattened

# correct_motion_lightcurve(flux, distance, thrust_indices, wavelet_params=None, mask_edges=True)

plt.figure(figsize=(12, 7))

# clipped_flux, flat_flux, outlier_mask = flatten_and_clip_outliers(flux_ape, frac=0.05, sigma=5.0)

filled_flux = correction_smoothing_lightcurve(flux_ape, distance < 0.25, window=35, sigma=3)
temp_flux = wavelet_denoise(filled_flux, wavelet=wavelet, level=3, keep='low', mode = mode)
plt.plot(flux_ape, label = 'Aperture', color = 'k')
plt.plot(temp_flux, label = f'Detrended', color = 'r')


# for level in levels:
#     try:
#         temp_flux = wavelet_denoise(filled_flux, wavelet=wavelet, level=level, keep='low', mode = mode)
#         plt.plot(temp_flux, label = f'wv: {wavelet}, mode: {mode}, lvl: {level}')
#     except:
#         print(f'wv: {wavelet}, mode: {mode}, lvl: {level}')

plt.legend(fontsize = 17)
plt.xlabel('Frame number', fontsize = 17)
plt.ylabel('Counts', fontsize = 17)
# plt.ylim(-150, None)
# plt.
# plt.savefig('/Users/zgl12/Modules/Kakapo/temp_mem/k2_pres_detrend.png', dpi = 900, bbox_inches='tight')
# plt.savefig('/Users/zgl12/Modules/Kakapo/temp_mem/mPlane_noise.png', dpi = 300)
plt.show()
